In [2]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

In [6]:
data = pd.read_parquet("dataset/cicids2017.csv")

In [10]:
data.head(100)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.000000,...,20,0.0,0.000,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.000000,...,20,0.0,0.000,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.000000,...,20,0.0,0.000,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.000000,...,20,0.0,0.000,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.000000,...,20,0.0,0.000,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,80,28466,1,1,6,6,6,6,6.0,0.000000,...,20,0.0,0.000,0,0,0.0,0.0,0,0,BENIGN
96,443,101027175,14,12,1029,991,460,0,73.5,150.040379,...,32,1048172.0,1296391.188,1964859,131485,49400000.0,12900000.0,58500000,40300000,BENIGN
97,53,330,2,2,74,198,37,37,37.0,0.000000,...,20,0.0,0.000,0,0,0.0,0.0,0,0,BENIGN
98,53,221,2,2,74,106,37,37,37.0,0.000000,...,20,0.0,0.000,0,0,0.0,0.0,0,0,BENIGN


In [11]:
data.shape

(225745, 79)

In [12]:
data.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Co

In [13]:
data['Label'].value_counts()

Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

In [14]:
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

In [15]:
data['Label'].value_counts()

Label
DDoS      128025
BENIGN     97686
Name: count, dtype: int64

In [16]:
data['Label'] = data['Label'].apply(lambda x: 0 if x == "BENIGN" else 1)

In [17]:
data

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225740,61374,61,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
225741,61378,72,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
225742,61375,75,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
225743,61323,48,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0


In [18]:
features = [
'Flow Duration',
'Total Fwd Packets',
'Total Backward Packets',
'Flow Bytes/s',
'Flow Packets/s',
'Packet Length Mean',
'Packet Length Std'
]

X = data[features]
y = data['Label']

In [19]:
X.head()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Flow Bytes/s,Flow Packets/s,Packet Length Mean,Packet Length Std
0,3,2,0,4.000000e+06,666666.66670,6.0,0.0
1,109,1,1,1.100917e+05,18348.62385,6.0,0.0
2,52,1,1,2.307692e+05,38461.53846,6.0,0.0
3,34,1,1,3.529412e+05,58823.52941,6.0,0.0
4,3,2,0,4.000000e+06,666666.66670,6.0,0.0


In [20]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Label, dtype: int64

In [21]:
X.shape

(225711, 7)

In [22]:
data[features + ['Label']].head()

,Flow Duration,Total Fwd Packets,Total Backward Packets,Flow Bytes/s,Flow Packets/s,Packet Length Mean,Packet Length Std,Label
0,3,2,0,4.000000e+06,666666.66670,6.0,0.0,0
1,109,1,1,1.100917e+05,18348.62385,6.0,0.0,0
2,52,1,1,2.307692e+05,38461.53846,6.0,0.0,0
3,34,1,1,3.529412e+05,58823.52941,6.0,0.0,0
4,3,2,0,4.000000e+06,666666.66670,6.0,0.0,0


In [23]:
from sklearn.preprocessing import StandardScaler

In [24]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [25]:
X_scaled[:5]

array([[-0.51525938, -0.18642364, -0.21020584,  0.20222141,  5.6681397 ,
        -0.91057496, -0.8552037 ],
       [-0.51525602, -0.25125787, -0.16424337, -0.02814857,  0.03568445,
        -0.91057496, -0.8552037 ],
       [-0.51525782, -0.25125787, -0.16424337, -0.02100175,  0.21042136,
        -0.91057496, -0.8552037 ],
       [-0.5152584 , -0.25125787, -0.16424337, -0.01376642,  0.3873222 ,
        -0.91057496, -0.8552037 ],
       [-0.51525938, -0.18642364, -0.21020584,  0.20222141,  5.6681397 ,
        -0.91057496, -0.8552037 ]])

In [26]:
from sklearn.ensemble import IsolationForest

In [27]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42
)

In [28]:
model.fit(X_scaled)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.1
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [29]:
predictions = model.predict(X_scaled)

In [30]:
predictions[:10]

array([-1,  1,  1,  1, -1,  1, -1,  1, -1, -1])

In [31]:
joblib.dump(model, "cyber_attack_model.pkl")

['cyber_attack_model.pkl']